<a href="https://colab.research.google.com/github/dev-gauravpingale/banking-data-platform-pyspark/blob/main/notebooks/02_Generate_Banking_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Banking Dataset Generation

## Objective

Generate realistic banking datasets that will be used throughout the project.

## Datasets

- Customers
- Accounts
- Transactions
- Branches
- Loans
- Cards

## Data Relationships

(Customers → Accounts → Transactions)

## Libraries

## Generate Branches

## Generate Customers

## Generate Accounts

## Generate Transactions

## Generate Loans

## Generate Cards

## Save to CSV

In [3]:
# ==========================================
# Project Configuration
# ==========================================

NUM_CUSTOMERS = 500
NUM_ACCOUNTS = 800
NUM_TRANSACTIONS = 50000
NUM_LOANS = 300
NUM_CARDS = 700

START_DATE = "2024-01-01"
END_DATE = "2025-12-31"

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

In [4]:
!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 25.0 MB/s eta 0:00:00


In [5]:
import os
import random
from datetime import datetime, timedelta

import pandas as pd
from faker import Faker

import numpy as np

In [3]:
fake = Faker("en_IN")
random.seed(42)
Faker.seed(42)

In [4]:
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/bronze", exist_ok=True)
os.makedirs("data/silver", exist_ok=True)
os.makedirs("data/gold", exist_ok=True)

In [5]:
branches = [
    ("Mumbai Main", "Mumbai", "Maharashtra"),
    ("Pune Central", "Pune", "Maharashtra"),
    ("Bengaluru MG Road", "Bengaluru", "Karnataka"),
    ("Hyderabad Banjara Hills", "Hyderabad", "Telangana"),
    ("Delhi Connaught Place", "New Delhi", "Delhi"),
    ("Chennai T Nagar", "Chennai", "Tamil Nadu"),
    ("Ahmedabad CG Road", "Ahmedabad", "Gujarat"),
    ("Kolkata Park Street", "Kolkata", "West Bengal"),
    ("Jaipur MI Road", "Jaipur", "Rajasthan"),
    ("Lucknow Hazratganj", "Lucknow", "Uttar Pradesh")
]

In [6]:
branch_records = []

for i, (name, city, state) in enumerate(branches, start=1):
    branch_records.append({
        "branch_id": i,
        "branch_name": name,
        "city": city,
        "state": state
    })

branches_df = pd.DataFrame(branch_records)

In [7]:
branches_df.head()

,branch_id,branch_name,city,state
0,1,Mumbai Main,Mumbai,Maharashtra
1,2,Pune Central,Pune,Maharashtra
2,3,Bengaluru MG Road,Bengaluru,Karnataka
3,4,Hyderabad Banjara Hills,Hyderabad,Telangana
4,5,Delhi Connaught Place,New Delhi,Delhi


In [8]:
branches_df.to_csv(
    "data/raw/branches.csv",
    index=False
)

In [9]:
os.listdir("data/raw")

['branches.csv']

In [10]:
customers = []

cities = [
    ("Mumbai", "Maharashtra"),
    ("Pune", "Maharashtra"),
    ("Bengaluru", "Karnataka"),
    ("Hyderabad", "Telangana"),
    ("Delhi", "Delhi"),
    ("Chennai", "Tamil Nadu"),
    ("Ahmedabad", "Gujarat"),
    ("Kolkata", "West Bengal"),
    ("Jaipur", "Rajasthan"),
    ("Lucknow", "Uttar Pradesh")
]

for customer_id in range(1, 501):

    city, state = random.choice(cities)

    customers.append({
        "customer_id": customer_id,
        "first_name": fake.first_name(),
        "last_name": fake.last_name(),
        "gender": random.choice(["Male", "Female"]),
        "date_of_birth": fake.date_of_birth(
            minimum_age=18,
            maximum_age=75
        ),
        "email": fake.email(),
        "phone": fake.msisdn()[:10],
        "city": city,
        "state": state,
        "created_date": fake.date_between(
            start_date="-5y",
            end_date="today"
        )
    })

In [12]:
customers_df = pd.DataFrame(customers)
customers_df.head()

,customer_id,first_name,last_name,gender,date_of_birth,email,phone,city,state,created_date
0,1,Isaac,Bakshi,Male,1993-07-09,udantdewan@example.net,8196001338,Pune,Maharashtra,2022-07-03
1,2,Oni,Kannan,Male,1976-07-25,abeer26@example.com,2351161559,Delhi,Delhi,2024-03-10
2,3,Kritika,Brar,Male,1982-07-12,rehaan10@example.net,4131647525,Hyderabad,Telangana,2025-01-07
3,4,Ekbal,Garg,Male,1981-06-28,nihalshere@example.com,8350305641,Pune,Maharashtra,2022-07-28
4,5,Xalak,Randhawa,Female,2001-10-28,qdhar@example.org,8849696532,Lucknow,Uttar Pradesh,2025-04-15


In [13]:
customers_df.describe(include="all")

,customer_id,first_name,last_name,gender,date_of_birth,email,phone,city,state,created_date
count,500.000000,500,500,500,500,500,500,500,500,500
unique,NaN,331,320,2,492,498,500,10,9,442
top,NaN,Gautami,Andra,Male,1985-09-20,vrama@example.org,3213580040,Pune,Maharashtra,2026-06-30
freq,NaN,5,6,252,2,2,1,62,92,3
mean,250.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,144.481833,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,125.750000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,250.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,375.250000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
#missing enmails
customers_df.loc[
    customers_df.sample(10, random_state=42).index,
    "email"
] = None

In [15]:
#duplicate customers
duplicate_rows = customers_df.sample(
    5,
    random_state=42
)

customers_df = pd.concat(
    [customers_df, duplicate_rows],
    ignore_index=True
)

In [16]:
#invalid phone numbers
customers_df.loc[
    customers_df.sample(5, random_state=10).index,
    "phone"
] = "123"

In [17]:
customers_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 505 entries, 0 to 504
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   customer_id    505 non-null    int64 
 1   first_name     505 non-null    object
 2   last_name      505 non-null    object
 3   gender         505 non-null    object
 4   date_of_birth  505 non-null    object
 5   email          490 non-null    object
 6   phone          505 non-null    object
 7   city           505 non-null    object
 8   state          505 non-null    object
 9   created_date   505 non-null    object
dtypes: int64(1), object(9)
memory usage: 39.6+ KB


In [18]:
customers_df.to_csv(
    "data/raw/customers.csv",
    index=False
)


In [19]:
os.listdir("data/raw")

['branches.csv', 'customers.csv']

Generate Accounts Dataset

In [20]:
account_types = [
    "Savings",
    "Current"
]

account_statuses = [
    "Active",
    "Dormant",
    "Closed"
]

In [21]:
accounts = []

for account_id in range(100001, 100801):

    customer = customers_df.sample(1).iloc[0]

    branch = branches_df.sample(1).iloc[0]

    accounts.append({

        "account_id": account_id,

        "customer_id": customer["customer_id"],

        "branch_id": branch["branch_id"],

        "account_type": random.choice(account_types),

        "balance": round(random.uniform(500, 500000), 2),

        "account_status": random.choices(
            account_statuses,
            weights=[85, 10, 5]
        )[0],

        "opened_date": fake.date_between(
            start_date="-10y",
            end_date="today"
        )

    })

In [22]:
accounts_df = pd.DataFrame(accounts)
accounts_df.head()

,account_id,customer_id,branch_id,account_type,balance,account_status,opened_date
0,100001,167,1,Current,493832.02,Active,2019-10-21
1,100002,259,1,Current,345311.60,Active,2022-08-21
2,100003,73,3,Current,100008.89,Active,2024-08-20
3,100004,305,2,Current,53773.31,Active,2019-03-17
4,100005,102,3,Current,287618.56,Closed,2018-03-19


In [23]:
accounts_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   account_id      800 non-null    int64  
 1   customer_id     800 non-null    int64  
 2   branch_id       800 non-null    int64  
 3   account_type    800 non-null    object 
 4   balance         800 non-null    float64
 5   account_status  800 non-null    object 
 6   opened_date     800 non-null    object 
dtypes: float64(1), int64(3), object(3)
memory usage: 43.9+ KB


In [24]:
duplicates = accounts_df.sample(
    5,
    random_state=42
)

accounts_df = pd.concat(
    [accounts_df, duplicates],
    ignore_index=True
)

Invalid Balance
Negative balance.

In [25]:
accounts_df.loc[
    accounts_df.sample(5, random_state=25).index,
    "balance"
] = -500

In [27]:
#invalid account type
accounts_df.loc[
    accounts_df.sample(5, random_state=100).index,
    "account_type"
] = "ABC"

In [28]:
#missing branch
accounts_df.loc[
    accounts_df.sample(5, random_state=100).index,
    "account_type"
] = "ABC"

In [29]:
accounts_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 805 entries, 0 to 804
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   account_id      805 non-null    int64  
 1   customer_id     805 non-null    int64  
 2   branch_id       805 non-null    int64  
 3   account_type    805 non-null    object 
 4   balance         805 non-null    float64
 5   account_status  805 non-null    object 
 6   opened_date     805 non-null    object 
dtypes: float64(1), int64(3), object(3)
memory usage: 44.2+ KB


In [30]:
accounts_df.to_csv(
    "data/raw/accounts.csv",
    index=False
)

In [31]:
os.listdir("data/raw")

['branches.csv', 'customers.csv', 'accounts.csv']

In [32]:
merchant_mapping = {
    "UPI": [
        "Swiggy",
        "Zomato",
        "Amazon",
        "Flipkart",
        "Uber",
        "Ola",
        "BigBasket",
        "Blinkit"
    ],

    "Card": [
        "Amazon",
        "Reliance Digital",
        "DMart",
        "Croma",
        "Lifestyle"
    ],

    "ATM": [
        "ABC Bank ATM"
    ],

    "Salary": [
        "Employer Payroll"
    ],

    "Interest": [
        "ABC Bank"
    ],

    "Cash Deposit": [
        "Branch Deposit"
    ],

    "NEFT": [
        "NEFT Transfer"
    ],

    "IMPS": [
        "IMPS Transfer"
    ]
}

# Generate Transactions Dataset

## Objective

Generate a realistic banking transaction dataset with approximately 50,000 records.

The generated data will be used throughout the Bronze, Silver and Gold layers of the Banking Data Platform project.

In [37]:
transaction_categories = {

    "UPI": (20, 5000),

    "Card": (100, 50000),

    "ATM": (100, 20000),

    "Salary": (30000, 250000),

    "Interest": (10, 5000),

    "Cash Deposit": (500, 200000),

    "NEFT": (1000, 500000),

    "IMPS": (100, 200000)

}

category_weights = {

    "UPI": 35,

    "Card": 25,

    "ATM": 15,

    "NEFT": 8,

    "IMPS": 7,

    "Salary": 5,

    "Interest": 3,

    "Cash Deposit": 2

}


merchant_mapping = {

    "UPI": [
        "Amazon",
        "Flipkart",
        "Swiggy",
        "Zomato",
        "Uber",
        "Ola",
        "BigBasket",
        "Blinkit"
    ],

    "Card": [
        "Reliance Digital",
        "DMart",
        "Croma",
        "Lifestyle"
    ],

    "ATM": [
        "ABC Bank ATM"
    ],

    "Salary": [
        "Employer Payroll"
    ],

    "Interest": [
        "ABC Bank"
    ],

    "Cash Deposit": [
        "Branch Deposit"
    ],

    "NEFT": [
        "NEFT Transfer"
    ],

    "IMPS": [
        "IMPS Transfer"
    ]
}

channels = {

    "UPI": "Mobile Banking",

    "Card": "POS",

    "ATM": "ATM",

    "Salary": "Internet Banking",

    "Interest": "Core Banking",

    "Cash Deposit": "Branch",

    "NEFT": "Internet Banking",

    "IMPS": "Mobile Banking"

}

In [38]:
from datetime import datetime, timedelta

start_datetime = datetime.strptime(
    START_DATE,
    "%Y-%m-%d"
)

end_datetime = datetime.strptime(
    END_DATE,
    "%Y-%m-%d"
)


def random_timestamp():

    delta = end_datetime - start_datetime

    random_seconds = random.randint(
        0,
        int(delta.total_seconds())
    )

    return start_datetime + timedelta(
        seconds=random_seconds
    )

In [41]:
random_timestamp()

datetime.datetime(2024, 1, 20, 10, 14, 3)

In [42]:
transactions = []

category_list = list(category_weights.keys())
category_probability = list(category_weights.values())

account_ids = accounts_df["account_id"].tolist()

for txn in range(1, NUM_TRANSACTIONS + 1):

    account_id = random.choice(account_ids)

    category = random.choices(
        category_list,
        weights=category_probability,
        k=1
    )[0]

    min_amt, max_amt = transaction_categories[category]

    amount = round(
        random.uniform(min_amt, max_amt),
        2
    )

    if category in ["Salary", "Interest", "Cash Deposit"]:
        transaction_type = "Credit"
    else:
        transaction_type = random.choices(
            ["Debit", "Credit"],
            weights=[90, 10],
            k=1
        )[0]

    merchant = random.choice(
        merchant_mapping[category]
    )
    channel = channels[category]
    status = random.choices(
        ["Success", "Failed", "Reversed"],
        weights=[97, 2, 1],
        k=1
    )[0]

    transactions.append({
        "transaction_id": f"TXN{txn:08d}",
        "account_id": account_id,
        "transaction_timestamp": random_timestamp(),
        "transaction_type": transaction_type,
        "transaction_category": category,
        "amount": amount,
        "merchant": merchant,
        "channel": channel,
        "status": status
    })

In [44]:
transactions_df = pd.DataFrame(transactions)
transactions_df.head()

,transaction_id,account_id,transaction_timestamp,transaction_type,transaction_category,amount,merchant,channel,status
0,TXN00000001,100760,2024-01-25 16:28:19,Debit,UPI,1131.59,Flipkart,Mobile Banking,Success
1,TXN00000002,100031,2025-06-28 17:38:42,Debit,UPI,1178.65,Zomato,Mobile Banking,Success
2,TXN00000003,100559,2024-05-04 00:15:55,Debit,Card,22515.53,Reliance Digital,POS,Success
3,TXN00000004,100715,2024-10-22 02:21:29,Debit,Card,13965.78,Croma,POS,Success
4,TXN00000005,100100,2025-02-20 12:00:58,Debit,Card,17263.39,Reliance Digital,POS,Success


In [45]:
transactions_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   transaction_id         50000 non-null  object        
 1   account_id             50000 non-null  int64         
 2   transaction_timestamp  50000 non-null  datetime64[ns]
 3   transaction_type       50000 non-null  object        
 4   transaction_category   50000 non-null  object        
 5   amount                 50000 non-null  float64       
 6   merchant               50000 non-null  object        
 7   channel                50000 non-null  object        
 8   status                 50000 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(6)
memory usage: 3.4+ MB


In [46]:
transactions_df["transaction_category"].value_counts()

,count
transaction_category,
UPI,17547
Card,12318
ATM,7556
NEFT,4096
IMPS,3447
Salary,2538
Interest,1501
Cash Deposit,997


In [47]:
transactions_df["status"].value_counts()

,count
status,
Success,48591
Failed,924
Reversed,485


In [48]:
transactions_df.to_csv(
    "data/raw/transactions.csv",
    index=False
)

In [49]:
os.listdir("data/raw")

['branches.csv', 'customers.csv', 'transactions.csv', 'accounts.csv']

In [1]:
"""
===========================================================
Banking Data Platform
Dataset Generator

Script : 01_generate_branches.py

Purpose:
Generate Branch Master dataset.

Output:
data/landing/branches.csv
===========================================================
"""

import os
import random
import pandas as pd

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

OUTPUT_FOLDER = "data/landing"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

random.seed(42)

# ---------------------------------------------------------
# Static Master Data
# ---------------------------------------------------------

branches = [
    ("Mumbai Fort", "Mumbai", "Maharashtra", "West", "West Zone"),
    ("Andheri East", "Mumbai", "Maharashtra", "West", "West Zone"),
    ("Pune Shivajinagar", "Pune", "Maharashtra", "West", "West Zone"),
    ("Bengaluru MG Road", "Bengaluru", "Karnataka", "South", "South Zone"),
    ("Chennai T Nagar", "Chennai", "Tamil Nadu", "South", "South Zone"),
    ("Hyderabad Banjara Hills", "Hyderabad", "Telangana", "South", "South Zone"),
    ("Delhi Connaught Place", "New Delhi", "Delhi", "North", "North Zone"),
    ("Jaipur C Scheme", "Jaipur", "Rajasthan", "North", "North Zone"),
    ("Kolkata Park Street", "Kolkata", "West Bengal", "East", "East Zone"),
    ("Ahmedabad Navrangpura", "Ahmedabad", "Gujarat", "West", "West Zone"),
]

manager_first_names = [
    "Rahul",
    "Amit",
    "Neha",
    "Priya",
    "Karan",
    "Sneha",
    "Rohit",
    "Pooja",
    "Arjun",
    "Anjali",
    "Vikram",
    "Deepak",
    "Meera",
    "Nikhil",
    "Shweta",
]

manager_last_names = [
    "Sharma",
    "Patel",
    "Gupta",
    "Kulkarni",
    "Joshi",
    "Verma",
    "Iyer",
    "Nair",
    "Reddy",
    "Singh",
]

# ---------------------------------------------------------
# Generate Dataset
# ---------------------------------------------------------

records = []

for i, branch in enumerate(branches, start=1):

    branch_name, city, state, region, zone = branch

    manager_name = (
        random.choice(manager_first_names)
        + " "
        + random.choice(manager_last_names)
    )

    ifsc = f"ABCB000{i:04d}"

    phone = "022-" + str(random.randint(10000000, 99999999))

    email = (
        branch_name.lower()
        .replace(" ", "")
        .replace(".", "")
        + "@abcbank.com"
    )

    records.append(
        {
            "branch_id": f"BR{i:03d}",
            "branch_name": branch_name,
            "city": city,
            "state": state,
            "region": region,
            "zone": zone,
            "ifsc_code": ifsc,
            "manager_name": manager_name,
            "branch_phone": phone,
            "branch_email": email,
        }
    )

# ---------------------------------------------------------
# Save CSV
# ---------------------------------------------------------

branches_df = pd.DataFrame(records)

output_path = os.path.join(
    OUTPUT_FOLDER,
    "branches.csv"
)

branches_df.to_csv(
    output_path,
    index=False
)

# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------

print("=" * 60)
print("Branches Dataset Generated Successfully")
print("=" * 60)
print(f"Rows Created : {len(branches_df)}")
print(f"Columns      : {len(branches_df.columns)}")
print(f"Output File  : {output_path}")
print("=" * 60)

print("\nSample Data\n")
print(branches_df.head())

Branches Dataset Generated Successfully
Rows Created : 10
Columns      : 10
Output File  : data/landing/branches.csv

Sample Data

  branch_id        branch_name       city        state region        zone  \
0     BR001        Mumbai Fort     Mumbai  Maharashtra   West   West Zone   
1     BR002       Andheri East     Mumbai  Maharashtra   West   West Zone   
2     BR003  Pune Shivajinagar       Pune  Maharashtra   West   West Zone   
3     BR004  Bengaluru MG Road  Bengaluru    Karnataka  South  South Zone   
4     BR005    Chennai T Nagar    Chennai   Tamil Nadu  South  South Zone   

     ifsc_code  manager_name  branch_phone                  branch_email  
0  ABCB0000001  Vikram Patel  022-13356886        mumbaifort@abcbank.com  
1  ABCB0000002  Deepak Joshi  022-42868828       andherieast@abcbank.com  
2  ABCB0000003   Priya Gupta  022-23756669  puneshivajinagar@abcbank.com  
3  ABCB0000004  Vikram Reddy  022-21668732   bengalurumgroad@abcbank.com  
4  ABCB0000005   Anjali Iyer  0

In [6]:
"""
===========================================================
Banking Data Platform
Dataset Generator

Script : 02_generate_customers.py

Purpose:
Generate Customer Master dataset.

Output:
data/landing/customers.csv
===========================================================
"""

import os
import random
import pandas as pd
from faker import Faker
from datetime import datetime

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

OUTPUT_FOLDER = "data/landing"
BRANCH_FILE = os.path.join(OUTPUT_FOLDER, "branches.csv")

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

fake = Faker("en_IN")

random.seed(42)
Faker.seed(42)

NUMBER_OF_CUSTOMERS = 500

# ---------------------------------------------------------
# Lookup Data
# ---------------------------------------------------------

occupations = [
    "Software Engineer",
    "Doctor",
    "Teacher",
    "Lawyer",
    "Business Owner",
    "Government Employee",
    "Sales Executive",
    "Chartered Accountant",
    "Mechanical Engineer",
    "Civil Engineer",
    "Student",
    "Retired",
]

segments = [
    "Retail",
    "Priority",
    "Premium"
]

kyc_statuses = [
    "Completed",
    "Pending"
]

risk_ratings = [
    "Low",
    "Medium",
    "High"
]

# ---------------------------------------------------------
# Read Branches
# ---------------------------------------------------------

branches_df = pd.read_csv(BRANCH_FILE)

cities = branches_df["city"].tolist()
states = branches_df["state"].tolist()

# ---------------------------------------------------------
# Generate Customers
# ---------------------------------------------------------

records = []

for i in range(1, NUMBER_OF_CUSTOMERS + 1):

    gender = random.choice(["Male", "Female"])

    if gender == "Male":
        first_name = fake.first_name_male()
    else:
        first_name = fake.first_name_female()

    last_name = fake.last_name()

    dob = fake.date_between(
        start_date="-65y",
        end_date="-18y"
    )

    city_index = random.randint(0, len(cities)-1)

    city = cities[city_index]
    state = states[city_index]

    created_date = fake.date_between(
        start_date="-5y",
        end_date="today"
    )

    email = (
        first_name.lower()
        + "."
        + last_name.lower()
        + str(random.randint(1,999))
        + "@gmail.com"
    )

    records.append({

        "customer_id": f"CUST{i:06d}",

        "first_name": first_name,

        "last_name": last_name,

        "gender": gender,

        "date_of_birth": dob,

        "email": email,

        "phone": fake.msisdn()[:10],

        "city": city,

        "state": state,

        "occupation": random.choice(occupations),

        "customer_segment": random.choices(
            segments,
            weights=[70,20,10],
            k=1
        )[0],

        "kyc_status": random.choices(
            kyc_statuses,
            weights=[95,5],
            k=1
        )[0],

        "risk_rating": random.choices(
            risk_ratings,
            weights=[75,20,5],
            k=1
        )[0],

        "created_date": created_date

    })

# ---------------------------------------------------------
# Create DataFrame
# ---------------------------------------------------------

customers_df = pd.DataFrame(records)

# ---------------------------------------------------------
# Save CSV
# ---------------------------------------------------------

output_path = os.path.join(
    OUTPUT_FOLDER,
    "customers.csv"
)

customers_df.to_csv(
    output_path,
    index=False
)

# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------

print("="*60)
print("Customers Dataset Generated Successfully")
print("="*60)

print(f"Rows Created : {len(customers_df)}")
print(f"Columns      : {len(customers_df.columns)}")
print(f"Output File  : {output_path}")

print("="*60)

print("\nSample Data\n")

print(customers_df.head())

Customers Dataset Generated Successfully
Rows Created : 500
Columns      : 14
Output File  : data/landing/customers.csv

Sample Data

  customer_id first_name last_name  gender date_of_birth  \
0  CUST000001      Daksh    Bakshi    Male    1996-05-13   
1  CUST000002     Hardik     Saini    Male    1971-11-16   
2  CUST000003      Kevin     Bassi    Male    1995-10-23   
3  CUST000004     Daksha       Rai  Female    1974-07-30   
4  CUST000005       Lopa     Kanda  Female    1993-09-09   

                       email       phone       city        state  \
0  daksh.bakshi760@gmail.com  1819600133     Mumbai  Maharashtra   
1  hardik.saini433@gmail.com  2654235116  Ahmedabad      Gujarat   
2   kevin.bassi204@gmail.com  1849593103    Kolkata  West Bengal   
3    daksha.rai778@gmail.com  2553419283     Mumbai  Maharashtra   
4     lopa.kanda95@gmail.com  3056413953     Mumbai  Maharashtra   

          occupation customer_segment kyc_status risk_rating created_date  
0     Business Owner

In [7]:
"""
===========================================================
Banking Data Platform
Dataset Generator

Script : 03_generate_accounts.py

Purpose:
Generate Account Master dataset.

Output:
data/landing/accounts.csv
===========================================================
"""

import os
import random
import pandas as pd
from faker import Faker

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

OUTPUT_FOLDER = "data/landing"

CUSTOMER_FILE = os.path.join(OUTPUT_FOLDER, "customers.csv")
BRANCH_FILE = os.path.join(OUTPUT_FOLDER, "branches.csv")

NUMBER_OF_ACCOUNTS = 800

random.seed(42)
Faker.seed(42)

# ---------------------------------------------------------
# Read Source Files
# ---------------------------------------------------------

customers_df = pd.read_csv(CUSTOMER_FILE)
branches_df = pd.read_csv(BRANCH_FILE)

customer_ids = customers_df["customer_id"].tolist()
branch_ids = branches_df["branch_id"].tolist()

# ---------------------------------------------------------
# Lookup Data
# ---------------------------------------------------------

account_types = [
    "Savings",
    "Current",
    "Salary",
    "NRE"
]

currencies = [
    "INR",
    "USD"
]

statuses = [
    "Active",
    "Dormant",
    "Closed"
]

interest_rates = {
    "Savings": 3.5,
    "Current": 0.0,
    "Salary": 3.0,
    "NRE": 4.0
}

# ---------------------------------------------------------
# Generate Accounts
# ---------------------------------------------------------

records = []

for i in range(1, NUMBER_OF_ACCOUNTS + 1):

    account_type = random.choices(
        account_types,
        weights=[70, 15, 10, 5],
        k=1
    )[0]

    customer_id = random.choice(customer_ids)

    opening_balance = round(
        random.uniform(1000, 500000),
        2
    )

    current_balance = round(
        opening_balance +
        random.uniform(-50000, 300000),
        2
    )

    current_balance = max(0, current_balance)

    opened_date = pd.Timestamp(
        random.choice(
            pd.date_range(
                "2019-01-01",
                "2025-01-01"
            )
        )
    ).date()

    records.append({

        "account_id": f"ACC{i:06d}",

        "customer_id": customer_id,

        "branch_id": random.choice(branch_ids),

        "account_type": account_type,

        "currency": random.choices(
            currencies,
            weights=[95,5],
            k=1
        )[0],

        "opening_balance": opening_balance,

        "current_balance": current_balance,

        "interest_rate": interest_rates[account_type],

        "account_status": random.choices(
            statuses,
            weights=[92,5,3],
            k=1
        )[0],

        "opened_date": opened_date

    })

# ---------------------------------------------------------
# Create DataFrame
# ---------------------------------------------------------

accounts_df = pd.DataFrame(records)

# ---------------------------------------------------------
# Save CSV
# ---------------------------------------------------------

output_path = os.path.join(
    OUTPUT_FOLDER,
    "accounts.csv"
)

accounts_df.to_csv(
    output_path,
    index=False
)

# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------

print("=" * 60)
print("Accounts Dataset Generated Successfully")
print("=" * 60)

print(f"Rows Created : {len(accounts_df)}")
print(f"Columns      : {len(accounts_df.columns)}")
print(f"Output File  : {output_path}")

print("=" * 60)

print("\nSample Data\n")

print(accounts_df.head())

Accounts Dataset Generated Successfully
Rows Created : 800
Columns      : 10
Output File  : data/landing/accounts.csv

Sample Data

  account_id customer_id branch_id account_type currency  opening_balance  \
0  ACC000001  CUST000013     BR002      Savings      INR        371033.70   
1  ACC000002  CUST000217     BR009      Savings      INR         16859.56   
2  ACC000003  CUST000360     BR001      Current      INR        272925.80   
3  ACC000004  CUST000143     BR002      Savings      INR         78584.27   
4  ACC000005  CUST000310     BR009      Current      INR        132995.91   

   current_balance  interest_rate account_status opened_date  
0        406745.85            3.5         Active  2020-07-25  
1             0.00            3.5         Active  2021-08-10  
2        300080.02            0.0         Active  2022-02-13  
3        363608.85            3.5         Active  2022-10-10  
4         98203.57            0.0        Dormant  2024-02-25  


In [8]:
"""
===========================================================
Banking Data Platform
Dataset Generator

Script : 04_generate_merchants.py

Purpose:
Generate Merchant Master dataset.

Output:
data/landing/merchants.csv
===========================================================
"""

import os
import random
import pandas as pd
from faker import Faker

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

OUTPUT_FOLDER = "data/landing"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

random.seed(42)
Faker.seed(42)
fake = Faker("en_IN")

NUMBER_OF_MERCHANTS = 100

# ---------------------------------------------------------
# Lookup Data
# ---------------------------------------------------------

merchant_categories = {
    "Grocery": [
        "DMart", "Reliance Fresh", "More", "Star Bazaar",
        "Big Bazaar", "Nature's Basket"
    ],
    "Fuel": [
        "Indian Oil", "HP Petrol Pump", "Bharat Petroleum",
        "Shell", "Nayara Energy"
    ],
    "Restaurant": [
        "Domino's", "McDonald's", "KFC",
        "Burger King", "Pizza Hut", "Subway"
    ],
    "Food Delivery": [
        "Swiggy", "Zomato"
    ],
    "Shopping": [
        "Amazon", "Flipkart", "Myntra",
        "Ajio", "Meesho"
    ],
    "Electronics": [
        "Croma", "Reliance Digital", "Vijay Sales"
    ],
    "Healthcare": [
        "Apollo Pharmacy", "MedPlus",
        "Fortis Hospital", "Max Hospital"
    ],
    "Entertainment": [
        "BookMyShow", "PVR Cinemas",
        "INOX", "Netflix", "Spotify"
    ],
    "Travel": [
        "IRCTC", "MakeMyTrip",
        "Goibibo", "Yatra"
    ],
    "Utilities": [
        "BESCOM", "MSEB", "TANGEDCO",
        "Airtel", "Jio", "Vi"
    ]
}

cities = [
    ("Mumbai", "Maharashtra"),
    ("Pune", "Maharashtra"),
    ("Bengaluru", "Karnataka"),
    ("Hyderabad", "Telangana"),
    ("Chennai", "Tamil Nadu"),
    ("Ahmedabad", "Gujarat"),
    ("Jaipur", "Rajasthan"),
    ("New Delhi", "Delhi"),
    ("Kolkata", "West Bengal"),
    ("Lucknow", "Uttar Pradesh")
]

# ---------------------------------------------------------
# Generate Merchants
# ---------------------------------------------------------

records = []

merchant_id = 1

while merchant_id <= NUMBER_OF_MERCHANTS:

    category = random.choice(list(merchant_categories.keys()))

    merchant_name = (
        random.choice(merchant_categories[category])
        + " "
        + fake.company_suffix()
    )

    city, state = random.choice(cities)

    records.append({

        "merchant_id": f"MER{merchant_id:04d}",

        "merchant_name": merchant_name,

        "merchant_category": category,

        "city": city,

        "state": state

    })

    merchant_id += 1

# ---------------------------------------------------------
# Create DataFrame
# ---------------------------------------------------------

merchants_df = pd.DataFrame(records)

# ---------------------------------------------------------
# Remove Duplicate Merchant Names
# ---------------------------------------------------------

merchants_df = merchants_df.drop_duplicates(
    subset=["merchant_name"]
).reset_index(drop=True)

# Reassign IDs after removing duplicates
merchants_df["merchant_id"] = [
    f"MER{i:04d}"
    for i in range(1, len(merchants_df) + 1)
]

# ---------------------------------------------------------
# Save CSV
# ---------------------------------------------------------

output_path = os.path.join(
    OUTPUT_FOLDER,
    "merchants.csv"
)

merchants_df.to_csv(
    output_path,
    index=False
)

# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------

print("=" * 60)
print("Merchants Dataset Generated Successfully")
print("=" * 60)

print(f"Rows Created : {len(merchants_df)}")
print(f"Columns      : {len(merchants_df.columns)}")
print(f"Output File  : {output_path}")

print("=" * 60)

print("\nSample Data\n")
print(merchants_df.head())

Merchants Dataset Generated Successfully
Rows Created : 86
Columns      : 5
Output File  : data/landing/merchants.csv

Sample Data

  merchant_id      merchant_name merchant_category       city        state
0     MER0001     Indian Oil Ltd              Fuel    Chennai   Tamil Nadu
1     MER0002         Swiggy Inc     Food Delivery  Bengaluru    Karnataka
2     MER0003  Nayara Energy Inc              Fuel       Pune  Maharashtra
3     MER0004         Airtel Ltd         Utilities     Mumbai  Maharashtra
4     MER0005          DMart LLC           Grocery  Hyderabad    Telangana


In [9]:
"""
===========================================================
Banking Data Platform
Dataset Generator

Script : 05A_generate_transactions.py

Purpose:
Generate Transaction Events

Output:
data/landing/transactions_part.csv
===========================================================
"""

import os
import random
import uuid
from datetime import timedelta

import pandas as pd

# ----------------------------------------------------------
# Configuration
# ----------------------------------------------------------

OUTPUT_FOLDER = "data/landing"

ACCOUNT_FILE = os.path.join(OUTPUT_FOLDER, "accounts.csv")
MERCHANT_FILE = os.path.join(OUTPUT_FOLDER, "merchants.csv")

OUTPUT_FILE = os.path.join(
    OUTPUT_FOLDER,
    "transactions_part.csv"
)

NUMBER_OF_TRANSACTIONS = 50000

random.seed(42)

# ----------------------------------------------------------
# Read Data
# ----------------------------------------------------------

accounts = pd.read_csv(ACCOUNT_FILE)
merchants = pd.read_csv(MERCHANT_FILE)

merchant_ids = merchants["merchant_id"].tolist()

# ----------------------------------------------------------
# Lookup Values
# ----------------------------------------------------------

transaction_types = [
    "Debit",
    "Credit"
]

channels = [
    "UPI",
    "ATM",
    "NEFT",
    "IMPS",
    "Debit Card",
    "Credit Card",
    "Cash",
    "Mobile Banking"
]

status_list = [
    "SUCCESS",
    "FAILED",
    "PENDING"
]

remarks_map = {
    "UPI": "UPI Payment",
    "ATM": "ATM Withdrawal",
    "NEFT": "NEFT Transfer",
    "IMPS": "IMPS Transfer",
    "Debit Card": "POS Purchase",
    "Credit Card": "Card Payment",
    "Cash": "Cash Deposit",
    "Mobile Banking": "Online Banking"
}

# ----------------------------------------------------------
# Maintain running balance
# ----------------------------------------------------------

balances = {}

for _, row in accounts.iterrows():
    balances[row["account_id"]] = float(row["current_balance"])

records = []

# ----------------------------------------------------------
# Generate Transactions
# ----------------------------------------------------------

for i in range(NUMBER_OF_TRANSACTIONS):

    account = accounts.sample(1).iloc[0]

    account_id = account["account_id"]

    balance = balances[account_id]

    channel = random.choice(channels)

    merchant = None

    if channel in [
        "UPI",
        "Debit Card",
        "Credit Card"
    ]:
        merchant = random.choice(merchant_ids)

    txn_type = random.choices(
        transaction_types,
        weights=[75,25]
    )[0]

    amount = round(
        random.uniform(50,20000),
        2
    )

    if txn_type == "Debit":

        if balance < amount:
            txn_type = "Credit"
            amount = round(
                random.uniform(
                    500,
                    50000
                ),
                2
            )

    if txn_type == "Debit":
        balance -= amount
    else:
        balance += amount

    balances[account_id] = round(balance,2)

    timestamp = (
        pd.Timestamp("2024-01-01")
        +
        timedelta(
            minutes=random.randint(
                0,
                525600
            )
        )
    )

    records.append({

        "reference_number": uuid.uuid4().hex[:16].upper(),

        "account_id": account_id,

        "transaction_timestamp": timestamp,

        "transaction_type": txn_type,

        "transaction_category": channel,

        "merchant_id": merchant,

        "amount": amount,

        "currency": account["currency"],

        "channel": channel,

        "balance_after_transaction": round(balance,2),

        "status": random.choices(
            status_list,
            weights=[97,2,1]
        )[0],

        "remarks": remarks_map[channel]

    })

# ----------------------------------------------------------
# Save
# ----------------------------------------------------------

transactions = pd.DataFrame(records)

transactions.to_csv(
    OUTPUT_FILE,
    index=False
)

print("="*60)
print("Transactions Part Generated")
print("="*60)

print("Rows :",len(transactions))
print("Output :",OUTPUT_FILE)

print("\nSample\n")

print(transactions.head())

Transactions Part Generated
Rows : 50000
Output : data/landing/transactions_part.csv

Sample

   reference_number account_id transaction_timestamp transaction_type  \
0  510B95E6A53844B2  ACC000259   2024-06-11 12:53:00            Debit   
1  605257DB9A4A44B7  ACC000158   2024-03-04 07:21:00            Debit   
2  6084FC7DA05A4158  ACC000362   2024-01-20 07:44:00            Debit   
3  4DCFD42241B8435D  ACC000033   2024-01-05 17:34:00            Debit   
4  D8BC75BDDC9D4A6E  ACC000487   2024-04-23 05:12:00            Debit   

  transaction_category merchant_id    amount currency channel  \
0                  ATM        None   5536.83      INR     ATM   
1                  ATM        None  17848.98      INR     ATM   
2                  UPI     MER0004   4691.58      INR     UPI   
3                 Cash        None  11805.85      INR    Cash   
4                 NEFT        None   6838.00      INR    NEFT   

   balance_after_transaction   status         remarks  
0                  1

In [10]:
"""
===========================================================
Banking Data Platform
Dataset Generator

Script : 05B_generate_transactions.py

Purpose:
Finalize Transactions Dataset

Input:
data/landing/transactions_part.csv

Output:
data/landing/transactions.csv
===========================================================
"""

import os
import pandas as pd

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

OUTPUT_FOLDER = "data/landing"

INPUT_FILE = os.path.join(
    OUTPUT_FOLDER,
    "transactions_part.csv"
)

OUTPUT_FILE = os.path.join(
    OUTPUT_FOLDER,
    "transactions.csv"
)

# ---------------------------------------------------------
# Read Intermediate File
# ---------------------------------------------------------

transactions = pd.read_csv(
    INPUT_FILE,
    parse_dates=["transaction_timestamp"]
)

# ---------------------------------------------------------
# Sort Transactions
# ---------------------------------------------------------

transactions = transactions.sort_values(
    by=[
        "transaction_timestamp",
        "account_id"
    ]
).reset_index(drop=True)

# ---------------------------------------------------------
# Generate Transaction IDs
# ---------------------------------------------------------

transactions.insert(
    0,
    "transaction_id",
    [
        f"TXN{i:09d}"
        for i in range(
            1,
            len(transactions) + 1
        )
    ]
)

# ---------------------------------------------------------
# Reorder Columns
# ---------------------------------------------------------

transactions = transactions[
    [
        "transaction_id",
        "reference_number",
        "account_id",
        "transaction_timestamp",
        "transaction_type",
        "transaction_category",
        "merchant_id",
        "amount",
        "currency",
        "channel",
        "balance_after_transaction",
        "status",
        "remarks"
    ]
]

# ---------------------------------------------------------
# Save Final Dataset
# ---------------------------------------------------------

transactions.to_csv(
    OUTPUT_FILE,
    index=False
)

# ---------------------------------------------------------
# Delete Intermediate File (Optional)
# ---------------------------------------------------------

try:
    os.remove(INPUT_FILE)
    print("Intermediate file deleted.")
except:
    pass

# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------

print("=" * 60)
print("Transactions Dataset Generated Successfully")
print("=" * 60)

print(f"Rows Created : {len(transactions)}")
print(f"Columns      : {len(transactions.columns)}")
print(f"Output File  : {OUTPUT_FILE}")

print("=" * 60)

print("\nSample Data\n")

print(transactions.head())

print("\nDataset Info\n")

print(transactions.info())

Intermediate file deleted.
Transactions Dataset Generated Successfully
Rows Created : 50000
Columns      : 13
Output File  : data/landing/transactions.csv

Sample Data

  transaction_id  reference_number account_id transaction_timestamp  \
0   TXN000000001  9E0860A978694A47  ACC000489   2024-01-01 00:55:00   
1   TXN000000002  A0EB44BC48964585  ACC000170   2024-01-01 01:27:00   
2   TXN000000003  B8C14AA9E0DE406B  ACC000253   2024-01-01 01:29:00   
3   TXN000000004  A000C09752714E62  ACC000366   2024-01-01 01:36:00   
4   TXN000000005  061519D63411439D  ACC000396   2024-01-01 01:38:00   

  transaction_type transaction_category merchant_id    amount currency  \
0            Debit       Mobile Banking         NaN  13708.38      INR   
1            Debit                 IMPS         NaN  17221.80      INR   
2            Debit                 NEFT         NaN  18907.31      USD   
3            Debit          Credit Card     MER0053   6106.28      INR   
4            Debit                

In [11]:
"""
===========================================================
Banking Data Platform
Dataset Generator

Script : 06_generate_loans.py

Purpose:
Generate Loan Master Dataset

Output:
data/landing/loans.csv
===========================================================
"""

import os
import random
import pandas as pd
from datetime import timedelta

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

OUTPUT_FOLDER = "data/landing"

ACCOUNT_FILE = os.path.join(OUTPUT_FOLDER, "accounts.csv")

OUTPUT_FILE = os.path.join(
    OUTPUT_FOLDER,
    "loans.csv"
)

NUMBER_OF_LOANS = 300

random.seed(42)

# ---------------------------------------------------------
# Read Accounts
# ---------------------------------------------------------

accounts = pd.read_csv(ACCOUNT_FILE)

# ---------------------------------------------------------
# Lookup Data
# ---------------------------------------------------------

loan_types = {
    "Home": 8.50,
    "Personal": 12.00,
    "Auto": 9.00,
    "Education": 8.00,
    "Business": 11.50
}

loan_amount_range = {
    "Home": (2000000, 10000000),
    "Personal": (100000, 1000000),
    "Auto": (400000, 2000000),
    "Education": (300000, 3000000),
    "Business": (500000, 5000000)
}

loan_status = [
    "Active",
    "Closed",
    "Defaulted"
]

tenure_options = [
    36,
    60,
    84,
    120,
    180,
    240
]

# ---------------------------------------------------------
# EMI Function
# ---------------------------------------------------------

def calculate_emi(principal, annual_rate, months):

    monthly_rate = annual_rate / 12 / 100

    emi = (
        principal
        * monthly_rate
        * (1 + monthly_rate) ** months
    ) / (
        ((1 + monthly_rate) ** months) - 1
    )

    return round(emi, 2)

# ---------------------------------------------------------
# Generate Loans
# ---------------------------------------------------------

records = []

sample_accounts = accounts.sample(
    NUMBER_OF_LOANS,
    replace=False,
    random_state=42
)

for i, row in sample_accounts.iterrows():

    loan_type = random.choice(
        list(loan_types.keys())
    )

    amount = round(
        random.uniform(
            loan_amount_range[loan_type][0],
            loan_amount_range[loan_type][1]
        ),
        2
    )

    tenure = random.choice(
        tenure_options
    )

    rate = loan_types[loan_type]

    emi = calculate_emi(
        amount,
        rate,
        tenure
    )

    disbursement = (
        pd.Timestamp("2020-01-01")
        +
        timedelta(
            days=random.randint(
                0,
                1800
            )
        )
    )

    maturity = (
        disbursement
        +
        pd.DateOffset(
            months=tenure
        )
    )

    paid_months = random.randint(
        0,
        tenure
    )

    outstanding = round(
        max(
            amount - (emi * paid_months),
            0
        ),
        2
    )

    records.append({

        "loan_id":
            f"LN{i+1:06d}",

        "customer_id":
            row["customer_id"],

        "account_id":
            row["account_id"],

        "loan_type":
            loan_type,

        "loan_amount":
            amount,

        "interest_rate":
            rate,

        "tenure_months":
            tenure,

        "emi":
            emi,

        "outstanding_balance":
            outstanding,

        "disbursement_date":
            disbursement.date(),

        "maturity_date":
            maturity.date(),

        "loan_status":
            random.choices(
                loan_status,
                weights=[90,7,3]
            )[0]

    })

# ---------------------------------------------------------
# Create DataFrame
# ---------------------------------------------------------

loans = pd.DataFrame(records)

# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

loans.to_csv(
    OUTPUT_FILE,
    index=False
)

# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------

print("="*60)
print("Loans Dataset Generated Successfully")
print("="*60)

print(f"Rows : {len(loans)}")
print(f"Columns : {len(loans.columns)}")

print("\nSample\n")

print(loans.head())

print("\nLoan Type Distribution\n")

print(
    loans["loan_type"]
    .value_counts()
)

Loans Dataset Generated Successfully
Rows : 300
Columns : 12

Sample

    loan_id customer_id account_id loan_type  loan_amount  interest_rate  \
0  LN000697  CUST000401  ACC000697      Home   2200086.04            8.5   
1  LN000668  CUST000127  ACC000668      Home   7413595.90            8.5   
2  LN000064  CUST000393  ACC000064      Home   2749561.92            8.5   
3  LN000534  CUST000435  ACC000534  Personal    744417.65           12.0   
4  LN000067  CUST000395  ACC000067  Business   1751858.19           11.5   

   tenure_months       emi  outstanding_balance disbursement_date  \
0             84  34841.63           1224520.40        2021-05-16   
1            180  73004.61                 0.00        2020-06-27   
2             60  56411.47            605926.06        2022-10-31   
3            240   8196.68                 0.00        2023-01-21   
4             36  57769.29           1174165.29        2024-04-03   

  maturity_date loan_status  
0    2028-05-16      Active 

In [12]:
"""
===========================================================
Banking Data Platform
Dataset Generator

Script : 07_generate_cards.py

Purpose:
Generate Debit & Credit Cards

Output:
data/landing/cards.csv
===========================================================
"""

import os
import random
import pandas as pd
from datetime import timedelta

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

OUTPUT_FOLDER = "data/landing"

ACCOUNT_FILE = os.path.join(
    OUTPUT_FOLDER,
    "accounts.csv"
)

OUTPUT_FILE = os.path.join(
    OUTPUT_FOLDER,
    "cards.csv"
)

random.seed(42)

# ---------------------------------------------------------
# Read Accounts
# ---------------------------------------------------------

accounts = pd.read_csv(ACCOUNT_FILE)

# ---------------------------------------------------------
# Lookup Values
# ---------------------------------------------------------

networks = [
    "Visa",
    "Mastercard",
    "RuPay"
]

statuses = [
    "Active",
    "Blocked",
    "Expired"
]

# ---------------------------------------------------------
# Generate Cards
# ---------------------------------------------------------

records = []

card_counter = 1

for _, account in accounts.iterrows():

    # -----------------------------
    # Debit Card (Every Account)
    # -----------------------------

    issue_date = (
        pd.Timestamp(account["opened_date"])
        + timedelta(days=random.randint(0,30))
    )

    expiry_date = issue_date + pd.DateOffset(years=5)

    records.append({

        "card_id":
            f"CARD{card_counter:06d}",

        "account_id":
            account["account_id"],

        "customer_id":
            account["customer_id"],

        "card_number_masked":
            f"XXXX-XXXX-XXXX-{random.randint(1000,9999)}",

        "card_type":
            "Debit",

        "network":
            random.choice(networks),

        "issue_date":
            issue_date.date(),

        "expiry_date":
            expiry_date.date(),

        "credit_limit":
            0,

        "available_limit":
            0,

        "card_status":
            random.choices(
                statuses,
                weights=[95,3,2]
            )[0]

    })

    card_counter += 1

    # -----------------------------
    # Credit Card Eligibility
    # -----------------------------

    if (
        account["current_balance"] >= 100000
        and
        account["account_status"] == "Active"
    ):

        limit = random.choice([
            50000,
            100000,
            200000,
            500000
        ])

        utilization = random.uniform(
            0,
            0.70
        )

        available = round(
            limit * (1-utilization),
            2
        )

        issue_date = (
            pd.Timestamp(account["opened_date"])
            + timedelta(
                days=random.randint(
                    30,
                    365
                )
            )
        )

        expiry_date = (
            issue_date
            + pd.DateOffset(years=5)
        )

        records.append({

            "card_id":
                f"CARD{card_counter:06d}",

            "account_id":
                account["account_id"],

            "customer_id":
                account["customer_id"],

            "card_number_masked":
                f"XXXX-XXXX-XXXX-{random.randint(1000,9999)}",

            "card_type":
                "Credit",

            "network":
                random.choice(networks),

            "issue_date":
                issue_date.date(),

            "expiry_date":
                expiry_date.date(),

            "credit_limit":
                limit,

            "available_limit":
                available,

            "card_status":
                random.choices(
                    statuses,
                    weights=[94,4,2]
                )[0]

        })

        card_counter += 1

# ---------------------------------------------------------
# Create DataFrame
# ---------------------------------------------------------

cards = pd.DataFrame(records)

# ---------------------------------------------------------
# Save CSV
# ---------------------------------------------------------

cards.to_csv(
    OUTPUT_FILE,
    index=False
)

# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------

print("="*60)
print("Cards Dataset Generated Successfully")
print("="*60)

print(f"Rows Created : {len(cards)}")
print(f"Columns      : {len(cards.columns)}")
print(f"Output File  : {OUTPUT_FILE}")

print("="*60)

print("\nCard Type Distribution\n")

print(cards["card_type"].value_counts())

print("\nNetwork Distribution\n")

print(cards["network"].value_counts())

print("\nSample Data\n")

print(cards.head())

Cards Dataset Generated Successfully
Rows Created : 1506
Columns      : 11
Output File  : data/landing/cards.csv

Card Type Distribution

card_type
Debit     800
Credit    706
Name: count, dtype: int64

Network Distribution

network
RuPay         508
Mastercard    500
Visa          498
Name: count, dtype: int64

Sample Data

      card_id account_id customer_id   card_number_masked card_type network  \
0  CARD000001  ACC000001  CUST000013  XXXX-XXXX-XXXX-2824     Debit    Visa   
1  CARD000002  ACC000001  CUST000013  XXXX-XXXX-XXXX-9935    Credit    Visa   
2  CARD000003  ACC000002  CUST000217  XXXX-XXXX-XXXX-1488     Debit    Visa   
3  CARD000004  ACC000003  CUST000360  XXXX-XXXX-XXXX-1434     Debit   RuPay   
4  CARD000005  ACC000003  CUST000360  XXXX-XXXX-XXXX-5557    Credit    Visa   

   issue_date expiry_date  credit_limit  available_limit card_status  
0  2020-08-14  2025-08-14             0             0.00      Active  
1  2020-10-15  2025-10-15        100000         84375.25

In [13]:
"""
===========================================================
Banking Data Platform
Dataset Generator

Script : 08_generate_calendar.py

Purpose:
Generate Calendar Dimension

Output:
data/landing/calendar.csv
===========================================================
"""

import os
import pandas as pd

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

OUTPUT_FOLDER = "data/landing"

OUTPUT_FILE = os.path.join(
    OUTPUT_FOLDER,
    "calendar.csv"
)

START_DATE = "2024-01-01"
END_DATE = "2025-12-31"

# ---------------------------------------------------------
# Generate Dates
# ---------------------------------------------------------

dates = pd.date_range(
    start=START_DATE,
    end=END_DATE,
    freq="D"
)

records = []

for dt in dates:

    records.append({

        "date_key":
            int(dt.strftime("%Y%m%d")),

        "date":
            dt.date(),

        "day":
            dt.day,

        "day_name":
            dt.day_name(),

        "day_of_week":
            dt.weekday() + 1,

        "week_of_year":
            dt.isocalendar().week,

        "month":
            dt.month,

        "month_name":
            dt.strftime("%B"),

        "quarter":
            f"Q{dt.quarter}",

        "year":
            dt.year,

        "is_weekend":
            dt.weekday() >= 5,

        "is_month_start":
            dt.is_month_start,

        "is_month_end":
            dt.is_month_end,

        "is_quarter_start":
            dt.is_quarter_start,

        "is_quarter_end":
            dt.is_quarter_end,

        "is_year_start":
            dt.is_year_start,

        "is_year_end":
            dt.is_year_end

    })

calendar = pd.DataFrame(records)

calendar.to_csv(
    OUTPUT_FILE,
    index=False
)

print("="*60)
print("Calendar Dataset Generated Successfully")
print("="*60)

print(f"Rows : {len(calendar)}")
print(f"Output : {OUTPUT_FILE}")

print(calendar.head())

Calendar Dataset Generated Successfully
Rows : 731
Output : data/landing/calendar.csv
   date_key        date  day   day_name  day_of_week  week_of_year  month  \
0  20240101  2024-01-01    1     Monday            1             1      1   
1  20240102  2024-01-02    2    Tuesday            2             1      1   
2  20240103  2024-01-03    3  Wednesday            3             1      1   
3  20240104  2024-01-04    4   Thursday            4             1      1   
4  20240105  2024-01-05    5     Friday            5             1      1   

  month_name quarter  year  is_weekend  is_month_start  is_month_end  \
0    January      Q1  2024       False            True         False   
1    January      Q1  2024       False           False         False   
2    January      Q1  2024       False           False         False   
3    January      Q1  2024       False           False         False   
4    January      Q1  2024       False           False         False   

   is_quarter_star

In [14]:
"""
===========================================================
Banking Data Platform
Dataset Generator

Script : 09_generate_exchange_rates.py

Purpose:
Generate Daily Exchange Rates

Output:
data/landing/exchange_rates.csv
===========================================================
"""

import os
import random
import pandas as pd

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

OUTPUT_FOLDER = "data/landing"

OUTPUT_FILE = os.path.join(
    OUTPUT_FOLDER,
    "exchange_rates.csv"
)

START_DATE = "2024-01-01"
END_DATE = "2025-12-31"

random.seed(42)

# ---------------------------------------------------------
# Currency Configuration
# ---------------------------------------------------------

currencies = {

    "INR": (1.00, 1.00),

    "USD": (82.00, 86.00),

    "EUR": (88.00, 95.00),

    "GBP": (102.00, 110.00)

}

# ---------------------------------------------------------
# Generate Exchange Rates
# ---------------------------------------------------------

dates = pd.date_range(
    START_DATE,
    END_DATE,
    freq="D"
)

records = []

for dt in dates:

    for currency, limits in currencies.items():

        if currency == "INR":

            rate = 1.00

        else:

            rate = round(
                random.uniform(
                    limits[0],
                    limits[1]
                ),
                4
            )

        records.append({

            "date": dt.date(),

            "currency": currency,

            "exchange_rate_to_inr": rate

        })

# ---------------------------------------------------------
# Create DataFrame
# ---------------------------------------------------------

exchange_rates = pd.DataFrame(records)

# ---------------------------------------------------------
# Save CSV
# ---------------------------------------------------------

exchange_rates.to_csv(
    OUTPUT_FILE,
    index=False
)

# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------

print("=" * 60)
print("Exchange Rates Dataset Generated Successfully")
print("=" * 60)

print(f"Rows Created : {len(exchange_rates)}")
print(f"Columns      : {len(exchange_rates.columns)}")
print(f"Output File  : {OUTPUT_FILE}")

print("=" * 60)

print("\nSample Data\n")

print(exchange_rates.head(10))

Exchange Rates Dataset Generated Successfully
Rows Created : 2924
Columns      : 3
Output File  : data/landing/exchange_rates.csv

Sample Data

         date currency  exchange_rate_to_inr
0  2024-01-01      INR                1.0000
1  2024-01-01      USD               84.5577
2  2024-01-01      EUR               88.1751
3  2024-01-01      GBP              104.2002
4  2024-01-02      INR                1.0000
5  2024-01-02      USD               82.8928
6  2024-01-02      EUR               93.1553
7  2024-01-02      GBP              107.4136
8  2024-01-03      INR                1.0000
9  2024-01-03      USD               85.5687
